---

# Training Auto-encoder

>Train auto-encoder with reconstruction loss.
>Audio is cropped to 2048 samples (~46ms at 44100 Hz).

---

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join('..'))

import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import IPython.display as ipd

from models.encoder import Encoder
from models.decoder import Decoder

CACHE_DIR = os.path.join('..', '.cache')
WEIGHTS_DIR = os.path.join('..', 'models', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

## Dataset

In [ ]:
y = np.load(os.path.join(CACHE_DIR, 'audio.npy'))
y = y[:, :2048]
print(f'Audio shape: {y.shape}')

## Build model

In [ ]:
with open(os.path.join('..', 'models', 'configs', 'encoder.json')) as f:
    enc_cfg = json.load(f)
with open(os.path.join('..', 'models', 'configs', 'ae-decoder.json')) as f:
    dec_cfg = json.load(f)

encoder = Encoder(**enc_cfg)
encoder.build(input_shape=(None, 2048))

decoder = Decoder(**dec_cfg)
decoder.build(input_shape=(None, enc_cfg['latent_dim']))

audio_input = layers.Input(shape=(2048,))
z = encoder(audio_input)
output = decoder(z)
ae = models.Model(audio_input, output)

ae.summary()

## Train

In [ ]:
ae.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
ae.fit(y, y, epochs=10, batch_size=32)

## Test

In [ ]:
y_test = y[0]
print('Original:')
ipd.display(ipd.Audio(data=y_test, rate=44100))

In [ ]:
latent = encoder.predict(np.array([y_test]), verbose=0)
y_recon = decoder.predict(latent, verbose=0)[0]

print('Reconstructed:')
ipd.display(ipd.Audio(data=y_recon, rate=44100))

## Save weights

In [ ]:
encoder.save_weights(os.path.join(WEIGHTS_DIR, 'encoder.h5'))
decoder.save_weights(os.path.join(WEIGHTS_DIR, 'decoder.h5'))
print('Saved.')

## Save latents for MLP training

In [ ]:
latents = encoder.predict(y, verbose=0)
np.save(os.path.join(CACHE_DIR, 'latents.npy'), latents)
print(f'Latents shape: {latents.shape}')